# 03 · Tuning & ablasi
Hanya Train + Val yang dimuat. Tuning: expanding-window CV + gap + 1-SE di Train. Val: konfirmasi finalis.

In [1]:
import sys; sys.path.insert(0, "../src")
import json, joblib
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import t3
from t3 import CFG, OUT, PRICE, SEED
from eval_utils import *

import time
from sklearn.decomposition import PCA

In [2]:
X = pd.read_parquet(OUT / "features.parquet")
blocks = json.loads((OUT / "blocks.json").read_text())
X = X[X.split != "test"]                                # Test sengaja tidak dimuat
tr, va = X[X.split == "train"], X[X.split == "val"]
EMB = CFG["emb_tags"]
SETS = {"A": PRICE, "B": PRICE + blocks["sent_lama"], "fb": PRICE + blocks["finbert"], **{t: PRICE + blocks[f"emb_{t}"] for t in EMB}}
TEXT = {"fb": blocks["finbert"], **{t: blocks[f"emb_{t}"] for t in EMB}}   # input model residual (late fusion)

def row(m, s, f, models, V=va):
    p = t3.predict(models, V); e = evaluate(V.target, p)
    return {"model": m, "set": s, "fusion": f, **{k: e[k] for k in ["MAE", "RMSE", "DirAcc", "PredStd"]}, "collapse": is_collapsed(p, V.target)}

print("Val | Prediksi 0 RMSE:", round(evaluate(va.target, np.zeros(len(va)))["RMSE"], 6), "| mayoritas DirAcc:", majority_baselines(va.target))

Val | Prediksi 0 RMSE: 0.003382 | mayoritas DirAcc: {'Selalu naik': 0.5340909090909091, 'Selalu turun': 0.4659090909090909, 'Mayoritas': 0.5340909090909091}


## Default (tanpa tuning, target distandardisasi)

In [3]:
D = {(m, s): t3.default_fit(m, c, tr, tr.target) for m in t3.MODELS for s, c in SETS.items()}
pd.DataFrame([row(m, s, "default", [mod]) for (m, s), mod in D.items()]).round(6)

,model,set,fusion,MAE,RMSE,DirAcc,PredStd,collapse
0,Ridge,A,default,0.002499,0.003373,0.522727,0.000601,False
1,Ridge,B,default,0.002544,0.003405,0.528409,0.000721,False
2,Ridge,fb,default,0.002500,0.003379,0.522727,0.000613,False
3,Ridge,bge_small,default,0.002537,0.003391,0.528409,0.000636,False
4,Ridge,bge_base,default,0.002543,0.003394,0.522727,0.000643,False
5,ElasticNet,A,default,0.002465,0.003379,0.534091,0.000000,True
6,ElasticNet,B,default,0.002465,0.003379,0.534091,0.000000,True
7,ElasticNet,fb,default,0.002465,0.003379,0.534091,0.000000,True
8,ElasticNet,bge_small,default,0.002465,0.003379,0.534091,0.000000,True
9,ElasticNet,bge_base,default,0.002465,0.003379,0.534091,0.000000,True


## Tuned: A, B, dan 6 sel C (3 sumber × early/late). Resumable lewat cache.

In [4]:
CACHE = OUT / "tuned.joblib"
R = joblib.load(CACHE) if CACHE.exists() else {}

def run(m, s, f, cols, **kw):
    if (m, s, f) not in R:
        R[(m, s, f)] = t3.tune(m, cols, tr, tr.target, **kw)
        joblib.dump(R, CACHE)

for m in t3.MODELS:
    t0 = time.time()
    for s in ("A", "B"): run(m, s, "early", SETS[s])
    for s in TEXT:
        run(m, s, "early", SETS[s])
        run(m, s, "late", TEXT[s], late_price=R[(m, "A", "early")]["models"][0])
    print(m, f"{time.time() - t0:.0f}s")

res = pd.DataFrame([{**row(m, s, f, r["models"]), "cv_rmse": r["cv_mse"] ** .5, "cv_se_mse": r["cv_se"]} for (m, s, f), r in R.items()])
res.round(6)

Ridge 20s
ElasticNet 17s
SVR 21s
GradientBoosting 58s


,model,set,fusion,MAE,RMSE,DirAcc,PredStd,collapse,cv_rmse,cv_se_mse
0,Ridge,A,early,0.002465,0.003379,0.534091,0.000007,True,0.003636,0.000002
1,Ridge,B,early,0.002465,0.003379,0.534091,0.000009,True,0.003636,0.000002
2,Ridge,fb,early,0.002465,0.003379,0.534091,0.000007,True,0.003636,0.000002
3,Ridge,fb,late,0.002464,0.003379,0.534091,0.000007,True,0.003650,0.000002
4,Ridge,bge_small,early,0.002490,0.003371,0.511364,0.000382,False,0.003622,0.000002
5,Ridge,bge_small,late,0.002480,0.003390,0.477273,0.000266,False,0.003644,0.000002
6,Ridge,bge_base,early,0.002486,0.003369,0.534091,0.000388,False,0.003620,0.000002
7,Ridge,bge_base,late,0.002476,0.003387,0.511364,0.000250,False,0.003644,0.000002
8,ElasticNet,A,early,0.002466,0.003373,0.545455,0.000068,True,0.003636,0.000002
9,ElasticNet,B,early,0.002466,0.003373,0.545455,0.000068,True,0.003639,0.000002


## Seleksi C* (aturan ditetapkan di muka)
Per model: 3 sel C terbaik menurut CV (yang tidak collapse) → pilih yang terbaik di Val. Model terbaik di antara C* dipakai untuk ablasi blok, kontrol negatif, dan urutan agregasi.

In [5]:
C = res[res.set.isin(TEXT) & ~res.collapse]
top = C.sort_values("cv_rmse").groupby("model").head(3)
final = top.loc[top.groupby("model")["RMSE"].idxmin()].set_index("model")
bm = final["RMSE"].idxmin()
bs, bf = final.loc[bm, ["set", "fusion"]]
etag = res[res.set.isin(EMB) & (res.fusion == "early") & (res.model == bm)].sort_values("cv_rmse").iloc[0]["set"]
print(f"Model terbaik: {bm} | C* = {bs}/{bf} | embedding untuk ablasi: {etag}")
final[["set", "fusion", "cv_rmse", "RMSE", "DirAcc"]].round(6)

Model terbaik: GradientBoosting | C* = bge_small/early | embedding untuk ablasi: bge_small


,set,fusion,cv_rmse,RMSE,DirAcc
model,,,,,
GradientBoosting,bge_small,early,0.003648,0.003363,0.539773
Ridge,bge_base,early,0.003620,0.003369,0.534091
SVR,bge_base,late,0.003655,0.003389,0.517045


## Ablasi blok (kumulatif, tiap langkah di-tuning)

In [6]:
steps = {"harga": PRICE, "+sent lama": blocks["sent_lama"], "+FinBERT": blocks["finbert"], f"+emb {etag}": blocks[f"emb_{etag}"],
         "+temporal F1-F3": blocks["F1"] + blocks["F2"] + blocks["F3"], "+ekstrem/cakupan F4-F5": blocks["F4"] + blocks["F5"],
         "+interaksi F6": blocks["F6"], "+turunan emb F7": blocks[f"F7_{etag}"], "+regime F8": blocks["F8"]}
abl, cols = {}, []
for k, c in steps.items():
    cols = cols + c
    abl[k] = R[(bm, "A", "early")] if k == "harga" else t3.tune(bm, list(cols), tr, tr.target)
pd.DataFrame([{"langkah": k, "cv_rmse": r["cv_mse"] ** .5, **row(bm, k, "early", r["models"])} for k, r in abl.items()]).drop(columns=["model", "set", "fusion"]).round(6)

,langkah,cv_rmse,MAE,RMSE,DirAcc,PredStd,collapse
0,harga,0.003687,0.002501,0.003362,0.551136,0.000582,False
1,+sent lama,0.003699,0.002527,0.003407,0.528409,0.000570,False
2,+FinBERT,0.003705,0.002518,0.003399,0.551136,0.000546,False
3,+emb bge_small,0.003659,0.002470,0.003366,0.539773,0.000224,False
4,+temporal F1-F3,0.003658,0.002454,0.003366,0.556818,0.000297,False
5,+ekstrem/cakupan F4-F5,0.003659,0.002454,0.003366,0.556818,0.000297,False
6,+interaksi F6,0.003663,0.002454,0.003366,0.556818,0.000297,False
7,+turunan emb F7,0.003664,0.002457,0.003368,0.556818,0.000300,False
8,+regime F8,0.003664,0.002457,0.003368,0.556818,0.000300,False


## Kontrol negatif: teks C* dengan baris diacak

In [7]:
tc = TEXT[bs]
perm = np.random.default_rng(SEED).permutation(len(X))
Xc = X.copy(); Xc[tc] = X[tc].to_numpy()[perm]
trc, vac = Xc[Xc.split == "train"], Xc[Xc.split == "val"]
late = {"late_price": R[(bm, "A", "early")]["models"][0]} if bf == "late" else {}
ctrl = t3.tune(bm, tc if bf == "late" else SETS[bs], trc, trc.target, **late)
pd.DataFrame([{"cv_rmse": r["cv_mse"] ** .5, **row(bm, n, bf, r["models"], V)} for n, r, V in
              [("C* asli", R[(bm, bs, bf)], va), ("kontrol (diacak)", ctrl, vac)]]).round(6)

,cv_rmse,model,set,fusion,MAE,RMSE,DirAcc,PredStd,collapse
0,0.003648,GradientBoosting,C* asli,early,0.002471,0.003363,0.539773,0.000236,False
1,0.003657,GradientBoosting,kontrol (diacak),early,0.002463,0.003360,0.551136,0.000254,False


## Urutan agregasi: PCA per artikel (fit di artikel train) lalu rata-rata harian, vs rata-rata harian lalu PCA

In [8]:
prices, articles, _ = t3.load_export()
train_end = prices.loc[prices.split == "train", "date"].max()
k = next(v for n, v in R[(bm, etag, "early")]["params"].items() if n.endswith("n_components"))
Xe = t3.load_emb(etag, articles)[0]
Z = PCA(k, random_state=SEED).fit(Xe[(articles.aligned_trading_date <= train_end).to_numpy()]).transform(Xe)
Zd = pd.DataFrame(Z, columns=[f"pc{i}" for i in range(k)]).groupby(articles.aligned_trading_date.values).mean().reindex(prices.date).fillna(0)
Zd.index = prices.index
X2 = X.join(Zd); tr2, va2 = X2[X2.split == "train"], X2[X2.split == "val"]
agg = t3.tune(bm, PRICE + list(Zd.columns), tr2, tr2.target)
pd.DataFrame([{"cv_rmse": r["cv_mse"] ** .5, **row(bm, n, "early", r["models"], V)} for n, r, V in
              [("rata-rata harian → PCA", R[(bm, etag, "early")], va), ("PCA artikel → rata-rata", agg, va2)]]).round(6)

,cv_rmse,model,set,fusion,MAE,RMSE,DirAcc,PredStd,collapse
0,0.003648,GradientBoosting,rata-rata harian → PCA,early,0.002471,0.003363,0.539773,0.000236,False
1,0.003684,GradientBoosting,PCA artikel → rata-rata,early,0.002546,0.003404,0.488636,0.000545,False


In [9]:
joblib.dump({"R": R, "D": D, "abl": abl, "ctrl": ctrl, "agg": agg, "Zd": Zd}, OUT / "tuned_all.joblib")
sel = {"final": final.reset_index()[["model", "set", "fusion"]].to_dict("records"), "best_model": bm, "etag": etag, "ctrl_cell": [bs, bf]}
(OUT / "selection.json").write_text(json.dumps(sel, indent=2))
res.to_csv(OUT / "val_results.csv", index=False)